## Global AI Job Market & Salary Trends 2025

### Research Question
What factors most significantly influence AI job salaries globally in 2025, and can we accurately predict salary levels using machine learning and distributed computing techniques?

### Dataset
- Source: Kaggle — Global AI Job Market & Salary Trends 2025
- Rows: 15,000 | Columns: 19
- Target variable: salary_usd (regression)

## Section 1: Data Description and Research Question

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv('ai_job_dataset.csv')

In [ ]:
print(f"Rows    : {len(data):,}")
print(f"Columns : {data.shape[1]}")
print(f"\nColumn types:")
print(data.dtypes)
data.head()

## Section 2: Data Preparation and Cleaning

In [ ]:
# Missing values and uniqueness snapshot
summary = pd.DataFrame({
    'dtype'     : data.dtypes.astype(str),
    'n_missing' : data.isna().sum(),
    'pct_missing': (data.isna().mean() * 100).round(2),
    'n_unique'  : data.nunique(),
})
summary.sort_values('n_missing', ascending=False)

In [ ]:
print("="*55)
print("   Section 2: Data Preparation and Cleaning")
print("="*55)

df_clean = data.copy()

# Step 1: Duplicates
dupes = df_clean.duplicated().sum()
print(f"\nDuplicate rows: {dupes} → No removal needed ✓")

# Step 2: Categorical consistency
cat_checks = {
    'experience_level': ['EN', 'MI', 'SE', 'EX'],
    'employment_type' : ['FT', 'PT', 'CT', 'FL'],
    'company_size'    : ['S', 'M', 'L'],
    'remote_ratio'    : [0, 50, 100]
}
print("\nCategorical consistency check:")
for col, expected in cat_checks.items():
    actual = sorted(df_clean[col].unique().tolist())
    unexpected = [x for x in actual if x not in expected]
    if unexpected:
        print(f"  {col}: unexpected → {unexpected}")
    else:
        print(f"  {col}: all valid ✓")

# Step 3: Numerical ranges
print(f"\nNumerical range check:")
print(f"  salary_usd:       "
      f"${df_clean['salary_usd'].min():,.0f} "
      f"— ${df_clean['salary_usd'].max():,.0f}")
print(f"  years_experience: "
      f"{df_clean['years_experience'].min()} "
      f"— {df_clean['years_experience'].max()}")
print(f"  Negative salaries: "
      f"{(df_clean['salary_usd'] < 0).sum()}")

# Step 4: Outlier detection
Q1  = df_clean['salary_usd'].quantile(0.25)
Q3  = df_clean['salary_usd'].quantile(0.75)
IQR = Q3 - Q1
outliers = df_clean[
    (df_clean['salary_usd'] < Q1 - 1.5*IQR) |
    (df_clean['salary_usd'] > Q3 + 1.5*IQR)
]
print(f"\nOutlier detection (IQR method):")
print(f"  Outliers found: {len(outliers)} rows "
      f"({len(outliers)/len(df_clean)*100:.1f}%)")
print(f"  Decision: RETAIN — represent genuine "
      f"executive salaries")

print(f"\n{'='*55}")
print(f"  Dataset confirmed clean: 15,000 rows, 19 columns")
print(f"  No records removed")
print(f"{'='*55}")

In [ ]:
# Parse dates and compute a days-open feature
for col in ['posting_date', 'application_deadline']:
  if col in data.columns:
    data[col] = pd.to_datetime(data[col], errors='coerce')

if {'posting_date', 'application_deadline'}.issubset(data.columns):
  data['days_open'] = (data['application_deadline'] - data['posting_date']).dt.days
  print(f"days_open - median: {data['days_open'].median():.0f} | p95: {data['days_open'].quantile(0.95):0f}")

data.describe(include='all').T.head(20)

## Section 3: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data['salary_usd'], bins=60, kde=True, ax=axes[0], color='#0f3460')
axes[0].set(title='Salary Distribution (USD)', xlabel='Annual Salary (USD)', ylabel='Count')
axes[0].axvline(data['salary_usd'].median(), color='#e94560', ls='--', label=f"median = ${data['salary_usd'].median():,.0f}")
axes[0].axvline(data['salary_usd'].mean(),   color='#faad14', ls='--', label=f"mean  = ${data['salary_usd'].mean():,.0f}")
axes[0].legend()

sns.histplot(np.log10(data['salary_usd']), bins=60, kde=True, ax=axes[1], color='#16213e')
axes[1].set(title='Log10(Salary) — a near-normal shape', xlabel='log10(Salary USD)', ylabel='Count')

plt.suptitle('Global AI Salary Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
print(f"Min:    ${data['salary_usd'].min():>10,.0f}")
print(f"25%:    ${data['salary_usd'].quantile(0.25):>10,.0f}")
print(f"Median: ${data['salary_usd'].median():>10,.0f}")
print(f"Mean:   ${data['salary_usd'].mean():>10,.0f}")
print(f"75%:    ${data['salary_usd'].quantile(0.75):>10,.0f}")
print(f"95%:    ${data['salary_usd'].quantile(0.95):>10,.0f}")
print(f"Max:    ${data['salary_usd'].max():>10,.0f}")

In [ ]:
EXP_MAP  = {'EN': 'Entry', 'MI': 'Mid', 'SE': 'Senior', 'EX': 'Executive'}
EXP_ORD  = ['Entry', 'Mid', 'Senior', 'Executive']
data['experience_level_full'] = data['experience_level'].map(EXP_MAP).fillna(data['experience_level'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order_present = [x for x in EXP_ORD if x in data['experience_level_full'].unique()]
sns.boxplot(data=data, x='experience_level_full', y='salary_usd', order=order_present,
            palette='viridis', ax=axes[0])
axes[0].set(title='Salary by Experience Level', xlabel='', ylabel='Salary (USD)')

exp_summary = data.groupby('experience_level_full')['salary_usd'].agg(['mean', 'median', 'count']).reindex(order_present)
exp_summary['median'].plot(kind='bar', color=['#9b59b6','#3498db','#27ae60','#e94560'], ax=axes[1])
axes[1].set(title='Median Salary by Experience', xlabel='', ylabel='Median Salary (USD)')
for i, v in enumerate(exp_summary['median']):
    axes[1].text(i, v, f'${v/1000:.0f}k', ha='center', va='bottom', fontweight='bold')
plt.xticks(rotation=0)

plt.suptitle('📈 Career Progression — AI Job Market', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()
exp_summary

In [ ]:
top_n = 15
top_countries = data['company_location'].value_counts().head(top_n).index

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

data['company_location'].value_counts().head(top_n).plot(kind='barh', color='#0f3460', ax=axes[0])
axes[0].set(title=f'Top {top_n} Countries by Job Postings', xlabel='# Postings', ylabel='')
axes[0].invert_yaxis()

country_salary = (data[data['company_location'].isin(top_countries)]
                  .groupby('company_location')['salary_usd']
                  .median().sort_values(ascending=True))
country_salary.plot(kind='barh', color='#e94560', ax=axes[1])
axes[1].set(title=f'Median Salary — Top {top_n} Countries', xlabel='Median Salary (USD)', ylabel='')
for i, v in enumerate(country_salary):
    axes[1].text(v, i, f' ${v/1000:.0f}k', va='center')

plt.suptitle('Geographic Market Structure', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Cross-border working: company vs employee location
data['cross_border'] = data['company_location'] != data['employee_residence']
xb = data['cross_border'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
xb.rename(index={True: 'Cross-border', False: 'Same country'}).plot(
    kind='bar', color=['#e94560', '#0f3460'], ax=axes[0])
axes[0].set(title='Share of Cross-Border AI Jobs', ylabel='% of postings')
for i, v in enumerate(xb.values):
    axes[0].text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')

sns.boxplot(data=data, x='cross_border', y='salary_usd', ax=axes[1],
            palette=['#0f3460', '#e94560'])
axes[1].set_xticklabels(['Same country', 'Cross-border'])
axes[1].set(title='Does Cross-Border Work Pay More?', xlabel='', ylabel='Salary (USD)')

plt.suptitle('Cross-Border Hiring Effect', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
REMOTE_MAP = {0: 'On-site', 50: 'Hybrid', 100: 'Fully Remote'}
data['remote_type'] = data['remote_ratio'].map(REMOTE_MAP).fillna('Unknown')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rcount = data['remote_type'].value_counts()
axes[0].pie(rcount.values, labels=rcount.index, autopct='%1.1f%%',
            colors=['#0f3460', '#e94560', '#faad14'], startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Remote / Hybrid / On-site Split', fontweight='bold')

sns.boxplot(data=data, x='remote_type', y='salary_usd',
            order=['On-site', 'Hybrid', 'Fully Remote'],
            palette=['#0f3460', '#faad14', '#e94560'], ax=axes[1])
axes[1].set(title='Salary by Work Arrangement', xlabel='', ylabel='Salary (USD)')

plt.suptitle('Remote Work — Frequency & Pay', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

data.groupby('remote_type')['salary_usd'].agg(['count', 'mean', 'median']).round(0)

In [ ]:
def tokenise_skills(text):
    if pd.isna(text):
        return []
    return [s.strip() for s in re.split(r'[,;/]', str(text)) if s.strip()]

data['skills_list'] = data['required_skills'].apply(tokenise_skills)
data['n_skills']    = data['skills_list'].apply(len)

all_skills = Counter()
for sk in data['skills_list']:
    all_skills.update(sk)

top_skills = all_skills.most_common(20)
top_df = pd.DataFrame(top_skills, columns=['skill', 'count'])
print(top_df)

In [ ]:
skill_salary = {}
for s, _ in top_skills:
    mask = data['skills_list'].apply(lambda lst: s in lst)
    skill_salary[s] = data.loc[mask, 'salary_usd'].median()
skill_sal_df = pd.DataFrame(
    [(k, all_skills[k], v) for k, v in skill_salary.items()],
    columns=['skill', 'count', 'median_salary']
).sort_values('median_salary', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_df.sort_values('count').plot(kind='barh', x='skill', y='count',
                                 color='#0f3460', legend=False, ax=axes[0])
axes[0].set(title='Top 20 Required Skills — Frequency', xlabel='# Postings')

skill_sal_df.plot(kind='barh', x='skill', y='median_salary',
                  color='#e94560', legend=False, ax=axes[1])
axes[1].set(title='Top 20 Skills — Median Salary', xlabel='Median Salary (USD)')

for i, v in enumerate(skill_sal_df['median_salary']):
    axes[1].text(v, i, f' ${v/1000:.0f}k', va='center', fontsize=9)

plt.suptitle('Skill Demand vs Skill Premium', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
SIZE_MAP = {'S': 'Small (<50)', 'M': 'Medium (50-250)', 'L': 'Large (>250)'}
data['company_size_full'] = data['company_size'].map(SIZE_MAP).fillna(data['company_size'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ind_top = data['industry'].value_counts().head(12).index
sns.boxplot(data=data[data['industry'].isin(ind_top)],
            x='salary_usd', y='industry',
            order=data[data['industry'].isin(ind_top)].groupby('industry')['salary_usd'].median().sort_values(ascending=False).index,
            palette='mako', ax=axes[0])
axes[0].set(title='Salary by Industry (top 12)', xlabel='Salary (USD)', ylabel='')

sns.violinplot(data=data, x='company_size_full', y='salary_usd',
               order=['Small (<50)', 'Medium (50-250)', 'Large (>250)'],
               palette=['#0f3460', '#faad14', '#e94560'], ax=axes[1])
axes[1].set(title='Salary by Company Size', xlabel='', ylabel='Salary (USD)')

plt.suptitle('Where the Money Is — Industry × Company Size', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
if 'posting_date' in data.columns and data['posting_date'].notna().any():
    data['posting_ym'] = data['posting_date'].dt.to_period('M').astype(str)
    monthly = data.groupby('posting_ym').agg(
        postings=('job_id', 'count'),
        median_salary=('salary_usd', 'median'),
    ).reset_index()

    fig, ax1 = plt.subplots(figsize=(15, 5))
    ax1.bar(monthly['posting_ym'], monthly['postings'], color='#0f3460', alpha=0.65, label='Postings')
    ax1.set_xlabel('Month'); ax1.set_ylabel('# Postings', color='#0f3460')
    ax1.tick_params(axis='y', labelcolor='#0f3460')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    ax2 = ax1.twinx()
    ax2.plot(monthly['posting_ym'], monthly['median_salary'],
             color='#e94560', marker='o', linewidth=2.5, label='Median Salary')
    ax2.set_ylabel('Median Salary (USD)', color='#e94560')
    ax2.tick_params(axis='y', labelcolor='#e94560')

    plt.title('Monthly Postings & Median Salary', fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('posting_date missing — skipping time-series.')

# **Feature Engineering for ML**

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_ml = data.copy()

# Target
df_ml['log_salary'] = np.log1p(df_ml['salary_usd'])

# Numerical features
NUMERIC_FEATS = [c for c in ['remote_ratio', 'years_experience', 'job_description_length',
                             'benefits_score', 'n_skills', 'days_open'] if c in df_ml.columns]
for c in NUMERIC_FEATS:
    df_ml[c] = pd.to_numeric(df_ml[c], errors='coerce')
    df_ml[c] = df_ml[c].fillna(df_ml[c].median())

# Categorical features — label-encode
CAT_FEATS = [c for c in ['experience_level', 'employment_type', 'company_location',
                         'company_size', 'employee_residence', 'education_required',
                         'industry', 'job_title'] if c in df_ml.columns]
encoders = {}
for c in CAT_FEATS:
    le = LabelEncoder()
    df_ml[c + '_enc'] = le.fit_transform(df_ml[c].astype(str))
    encoders[c] = le

# One-hot the 5 most common skills as extra features
top5 = [s for s, _ in all_skills.most_common(5)]
for s in top5:
    df_ml[f'skill_{s.replace(" ","_")}'] = df_ml['skills_list'].apply(lambda lst: int(s in lst))

FEATURES = NUMERIC_FEATS + [c + '_enc' for c in CAT_FEATS] + [f'skill_{s.replace(" ","_")}' for s in top5]
FEATURES = [f for f in FEATURES if f in df_ml.columns]
print(f"{len(FEATURES)} features ready")
print(FEATURES)

### Section 3.1: Unsupervised Learning — PCA and DBSCAN

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN

# Select the features used for modeling, excluding the target variable 'log_salary'
X_clustering = df_ml[FEATURES]

# Scale the data before applying PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clustering)

# Apply PCA
pca = PCA(n_components=2) # Reduce to 2 dimensions for visualization
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance ratio by PCA components: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.2f}")

# Apply DBSCAN with updated parameters
dbscan = DBSCAN(eps=0.15, min_samples=5) # Updated 'eps' based on k-distance graph
clusters = dbscan.fit_predict(X_pca)

df_ml['pca1'] = X_pca[:, 0]
df_ml['pca2'] = X_pca[:, 1]
df_ml['dbscan_cluster'] = clusters

# Visualize DBSCAN results (only for 2D PCA)
plt.figure(figsize=(10, 7))
sns.scatterplot(x='pca1', y='pca2', hue='dbscan_cluster', data=df_ml, palette='viridis', s=50, legend='full')
plt.title('DBSCAN Clustering on PCA-transformed Data (Updated parameters)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

print(f"Number of clusters found by DBSCAN: {len(np.unique(clusters)) - (1 if -1 in clusters else 0)}")
print(f"Number of noise points (assigned to -1): {(clusters == -1).sum()}")

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Determine k for k-distance graph. A common heuristic for min_samples is 2 * number of features.
k = 5 # Using 5 as a starting point, same as the previous run's min_samples

neigh = NearestNeighbors(n_neighbors=k)
neigh.fit(X_pca)
distances, indices = neigh.kneighbors(X_pca)

# Sort distances to the k-th nearest neighbor
distances = np.sort(distances[:, k-1], axis=0)

plt.figure(figsize=(12, 6))
plt.plot(distances)
plt.title(f'K-distance Graph (k={k})')
plt.xlabel('Points sorted by distance')
plt.ylabel(f'Distance to {k}-th nearest neighbor')
plt.grid(True)
plt.show()

print(f"The k-distance graph helps to visually identify a suitable 'eps' value. Look for the 'elbow' where the curve sharply changes. \nFor 'min_samples', a general guideline is to choose a value at least 2 * number of dimensions (features), so for 2 PCA components, a value of 5 is a reasonable starting point.")

**Salary Prediction - Regression Models**

## Section 4: Machine Learning Prediction

In [ ]:
from sklearn.tree import DecisionTreeRegressor as SklearnDT
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

X = df_ml[FEATURES].values
y = df_ml['log_salary'].values

# Scale features
scaler_ml = StandardScaler()
X_scaled_ml = scaler_ml.fit_transform(X)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled_ml, y, test_size=0.2, random_state=42
)

# Shared results list
ml_results = []
print("Setup complete — ready to train models!")

SKLearn - Neural Network (MLP)

In [ ]:
# ════════════════════════════════════════
# Neural Network (MLP)
# ════════════════════════════════════════

from sklearn.model_selection import GridSearchCV

# Train base model
nn = MLPRegressor(hidden_layer_sizes=(100, 50),
                  max_iter=500, random_state=42)
nn.fit(X_tr, y_tr)
pred_log = nn.predict(X_te)
pred_usd = np.expm1(pred_log)
true_usd = np.expm1(y_te)

print("Member 5 — Neural Network (before tuning):")
print(f"  R²:       {r2_score(y_te, pred_log):.4f}")
print(f"  MAE USD:  ${mean_absolute_error(true_usd, pred_usd):,.0f}")
print(f"  RMSE USD: ${np.sqrt(mean_squared_error(true_usd, pred_usd)):,.0f}")

# Hyperparameter tuning
nn_params = {
    'hidden_layer_sizes': [(100, 50), (100, 100)],
    'max_iter'          : [300, 500]
}
nn_grid = GridSearchCV(
    MLPRegressor(random_state=42),
    nn_params, cv=3, scoring='r2', n_jobs=-1
)
nn_grid.fit(X_tr, y_tr)
best_nn = nn_grid.best_estimator_
pred_log_tuned = best_nn.predict(X_te)
pred_usd_tuned = np.expm1(pred_log_tuned)

print(f"\nMember 5 — Neural Network (after tuning):")
print(f"  Best params: {nn_grid.best_params_}")
print(f"  R²:          {r2_score(y_te, pred_log_tuned):.4f}")
print(f"  MAE USD:     ${mean_absolute_error(true_usd, pred_usd_tuned):,.0f}")
print(f"  RMSE USD:    ${np.sqrt(mean_squared_error(true_usd, pred_usd_tuned)):,.0f}")

ml_results.append({
    'Member': 'Member 5',
    'Model' : 'Neural Network',
    'R²'    : round(r2_score(y_te, pred_log_tuned), 4),
    'MAE'   : round(mean_absolute_error(true_usd, pred_usd_tuned), 0),
    'RMSE'  : round(np.sqrt(mean_squared_error(true_usd, pred_usd_tuned)), 0)
})

## **Performing HPC Ml Models in PySpark on the same data set to find the optimal results**

Spark Session

## Section 5: High Performance Computational Implementation

In [ ]:
# Environment setup

!pip install pyspark

from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("AI_Salary_Prediction") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()
sc = spark.sparkContext

print("Spark version:", spark.version)
print("Default parallelism:", sc.defaultParallelism)

Load data into Spark

In [ ]:
spark_df = spark.read.csv('ai_job_dataset.csv',
                           header=True,
                           inferSchema=True)

print(f"Rows: {spark_df.count()}")
print(f"Columns: {len(spark_df.columns)}")
spark_df.printSchema()

Build preprocessing pipeline (Regression)

In [ ]:
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

# Categorical columns to encode
categoricalColumns = [
    'experience_level', 'employment_type',
    'company_size', 'education_required',
    'industry', 'job_title', 'company_location'
]

# Numerical columns
numericCols = [
    'years_experience', 'remote_ratio',
    'job_description_length', 'benefits_score'
]

stages = []

# StringIndexer for each categorical column
for cat_col in categoricalColumns:
    indexer = StringIndexer(
        inputCol=cat_col,
        outputCol=cat_col + '_idx',
        handleInvalid='keep'
    )
    stages += [indexer]

# Assemble all features into one vector
assembler_inputs = [c + '_idx' for c in categoricalColumns] + numericCols
assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol='raw_features',
    handleInvalid='keep'
)
stages += [assembler]

# Scale features
scaler = StandardScaler(
    inputCol='raw_features',
    outputCol='features',
    withStd=True,
    withMean=True
)
stages += [scaler]

print("Pipeline stages defined:", len(stages))

Fit pipeline and split data

In [ ]:
# Fit preprocessing pipeline
pipeline = Pipeline(stages=stages)
processed_df = pipeline.fit(spark_df).transform(spark_df)

# Select only features and target for training
model_df = processed_df.select('features',
                                col('salary_usd').alias('label'))

# 80/20 train test split — same as sklearn
train, test = model_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train.count()}")
print(f"Testing rows:  {test.count()}")

Define evaluation function for regression

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate_regression(predictions):
    # RMSE
    evaluator_rmse = RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    )
    # MAE
    evaluator_mae = RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='mae'
    )
    # R²
    evaluator_r2 = RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='r2'
    )

    rmse = evaluator_rmse.evaluate(predictions)
    mae  = evaluator_mae.evaluate(predictions)
    r2   = evaluator_r2.evaluate(predictions)

    return rmse, mae, r2

results = []
print("Evaluation function defined!")

PySpark - RF + PCA

In [ ]:
# PySpark RF Regressor using PCA features
from pyspark.ml.feature import PCA
from pyspark.ml.regression import RandomForestRegressor as SparkRF
from pyspark.sql.functions import col

# Build PCA on features
pca = PCA(k=5, inputCol='features', outputCol='pca_features')
pca_model = pca.fit(processed_df)
pca_df = pca_model.transform(processed_df)

print("PCA Explained Variance:")
for i, v in enumerate(pca_model.explainedVariance):
    print(f"  Component {i+1}: {v*100:.1f}%")
print(f"  Total: {sum(pca_model.explainedVariance)*100:.1f}%")

# Add salary label
pca_df = pca_df.withColumn('label', col('salary_usd'))

# Split
pca_train, pca_test = pca_df.select(
    'pca_features', 'label'
).randomSplit([0.8, 0.2], seed=42)

# Train RF on PCA features
rf_pca = SparkRF(
    featuresCol='pca_features',
    labelCol='label',
    numTrees=100,
    maxDepth=10,
    seed=42
)

start = time.time()
rf_pca_model = rf_pca.fit(pca_train)
pca_time = time.time() - start

pca_preds = rf_pca_model.transform(pca_test)

# Evaluate
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(labelCol='label',
                  predictionCol='prediction', metricName='rmse')
evaluator_mae  = RegressionEvaluator(labelCol='label',
                  predictionCol='prediction', metricName='mae')
evaluator_r2   = RegressionEvaluator(labelCol='label',
                  predictionCol='prediction', metricName='r2')

print("=" * 45)
print("  PySpark RF + PCA Features (Member 5)")
print("=" * 45)
print(f"  MAE:  ${evaluator_mae.evaluate(pca_preds):,.0f}")
print(f"  RMSE: ${evaluator_rmse.evaluate(pca_preds):,.0f}")
print(f"  R²:    {evaluator_r2.evaluate(pca_preds):.4f}")
print(f"  Time:  {pca_time:.2f}s")
print("=" * 45)


Hyperparameter tuning

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Tune Random Forest — smaller grid to avoid memory issues
rf_tune = SparkRF(
    featuresCol='features',
    labelCol='label',
    seed=42
)

# Reduced grid — only 4 combinations instead of 9
paramGrid = ParamGridBuilder() \
    .addGrid(rf_tune.numTrees, [50, 100]) \
    .addGrid(rf_tune.maxDepth, [5, 10]) \
    .build()

evaluator_r2 = RegressionEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='r2'
)

# Reduced to 2 folds instead of 3 to save memory
cv = CrossValidator(
    estimator=rf_tune,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_r2,
    numFolds=2,
    seed=42,
    parallelism=1  # reduced from 2 to 1 to save memory
)

print("Starting hyperparameter tuning...")
print("Testing 4 combinations with 2-fold CV (8 total fits)")
start = time.time()
cv_model = cv.fit(train)
cv_time  = time.time() - start

best_preds = cv_model.bestModel.transform(test)
rmse, mae, r2 = evaluate_regression(best_preds)

print(f"Tuning completed in {cv_time:.2f} seconds")
print(f"\nBest Random Forest (Tuned):")
print(f"  RMSE: ${rmse:,.0f}")
print(f"  MAE:  ${mae:,.0f}")
print(f"  R²:    {r2:.4f}")
print(f"  Best numTrees: {cv_model.bestModel.getNumTrees}")
print(f"  Best maxDepth: {cv_model.bestModel.getMaxDepth()}")

PySpark PCA

In [ ]:
from pyspark.ml.feature import PCA

pca = PCA(k=5, inputCol='features', outputCol='pca_features')
pca_model = pca.fit(processed_df)

print("PCA Explained Variance per component:")
for i, v in enumerate(pca_model.explainedVariance):
    print(f"  Component {i+1}: {v:.4f} ({v*100:.1f}%)")
print(f"  Total explained: "
      f"{sum(pca_model.explainedVariance)*100:.1f}%")

PySpark K-Means

In [ ]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(k=3, seed=42, featuresCol='features')
km_model = kmeans.fit(processed_df)
km_preds = km_model.transform(processed_df)

print("K-Means Cluster Sizes:")
km_preds.groupBy('prediction').count().orderBy('prediction').show()

print(f"Within-cluster sum of squares: "
      f"{km_model.summary.trainingCost:.2f}")